# Semantic N3 Planner - Role-Annotated hasInput

This notebook demonstrates the **semantic planner** that generates PolicyCheckers with **role-annotated hasInput** from ODRL policies.

## Features
- 🔄 Transforms ODRL policies → PolicyChecker operations
- 🎯 Role-annotated hasInput (no ambiguity with multiple parameters)
- 📊 N3 reasoning for policy transformation
- 🧪 Modular design for testing and extension

## Architecture
```
ODRL Policy (SDM) 
  ↓ [N3 Transformation Rules]
PolicyChecker Graph (role-annotated hasInput)
  ↓ [Executor]
Validation Results
```

In [ ]:
import sys
import subprocess
import tempfile
from pathlib import Path
from rdflib import Graph, Namespace, RDF, URIRef, BNode
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Define namespaces
tb = Namespace("http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#")
ab = Namespace("http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#")
odrl = Namespace("http://www.w3.org/ns/odrl/2/")

print("✓ Imports loaded")
print(f"✓ Namespaces: tb, ab, odrl")

In [ ]:
class EYEReasoner:
    """Wrapper for EYE reasoner to apply N3 rules"""
    
    def __init__(self):
        self.logger = logging.getLogger(f"{__name__}.EYEReasoner")
    
    def apply_rules(self, input_graph: Graph, rules_dir: Path) -> Graph:
        """Apply N3 rules from directory to input graph"""
        self.logger.info(f"Applying N3 rules from {rules_dir}")
        
        # Find all N3 rule files
        rule_files = sorted(rules_dir.glob("*.n3"))
        self.logger.info(f"Found {len(rule_files)} rule files:")
        for rf in rule_files:
            self.logger.info(f"  - {rf.name}")
        
        # Write input graph to temp file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.n3', delete=False) as f:
            input_graph.serialize(f, format='n3')
            input_file = f.name
        
        # Create query to extract all triples
        query = """
        @prefix : <http://example.org/>.
        {?s ?p ?o} => {?s ?p ?o}.
        """
        with tempfile.NamedTemporaryFile(mode='w', suffix='.n3', delete=False) as f:
            f.write(query)
            query_file = f.name
        
        try:
            # Build EYE command
            cmd = ['eye', input_file] + [str(rf) for rf in rule_files]
            cmd += ['--query', query_file, '--nope']  # --nope = no proof output
            
            cmd_str = ' '.join(cmd)
            self.logger.info(f"Running EYE: {cmd_str}")
            
            # Run EYE
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            
            # Parse output
            output_graph = Graph()
            output_graph.parse(data=result.stdout, format='n3')
            
            self.logger.info(f"EYE output {len(output_graph)} total triples")
            
            # Filter to domain triples only (remove reasoning metadata)
            domain_graph = Graph()
            for s, p, o in output_graph:
                if not str(s).startswith('https://eyereasoner.github.io'):
                    domain_graph.add((s, p, o))
            
            self.logger.info(f"Filtered to {len(domain_graph)} domain triples")
            
            return domain_graph
            
        finally:
            # Cleanup temp files
            Path(input_file).unlink()
            Path(query_file).unlink()

# Test EYE reasoner
reasoner = EYEReasoner()
print("✓ EYE Reasoner initialized")

In [ ]:
class SemanticN3Planner:
    """Generates PolicyCheckers using N3 rules with role-annotated hasInput"""
    
    def __init__(self, rules_dir: str = "rules_n3_semantic"):
        self.logger = logging.getLogger(f"{__name__}.SemanticN3Planner")
        self.rules_dir = Path(__file__).parent / rules_dir if not Path(rules_dir).is_absolute() else Path(rules_dir)
        self.reasoner = EYEReasoner()
        
        if not self.rules_dir.exists():
            raise FileNotFoundError(f"Rules directory not found: {self.rules_dir}")
        
        self.logger.info(f"Using rules directory: {self.rules_dir}")
    
    def load_sdm(self, sdm_path: str) -> Graph:
        """Load Semantic Data Model"""
        self.logger.info(f"Loading SDM from {sdm_path}")
        graph = Graph()
        graph.parse(sdm_path, format='turtle')
        self.logger.info(f"Loaded {len(graph)} triples from SDM")
        return graph
    
    def generate_policy_checkers(self, sdm_graph: Graph, dataset_id: str) -> Graph:
        """Generate PolicyCheckers for a specific dataset"""
        self.logger.info(f"Generating PolicyCheckers for: {dataset_id}")
        
        # Apply N3 transformation rules
        result_graph = self.reasoner.apply_rules(sdm_graph, self.rules_dir)
        
        # Extract PolicyCheckers for this dataset
        policy_checker_graph = Graph()
        
        # Bind namespaces
        policy_checker_graph.bind('tbox', tb)
        policy_checker_graph.bind('abox', ab)
        policy_checker_graph.bind('odrl', odrl)
        
        # Find all PolicyCheckers for this dataset
        dataset_uri = ab[dataset_id]
        
        for pc in result_graph.subjects(RDF.type, tb.PolicyChecker):
            validates = result_graph.value(pc, tb.validates)
            if validates == dataset_uri:
                self.logger.info(f"Found PolicyChecker: {pc}")
                self._add_policy_checker(pc, result_graph, policy_checker_graph)
        
        return policy_checker_graph
    
    def _add_policy_checker(self, pc_uri, source_graph, target_graph):
        """Add PolicyChecker and its operation chain to target graph"""
        # Add PolicyChecker triples
        for p, o in source_graph.predicate_objects(pc_uri):
            target_graph.add((pc_uri, p, o))
        
        # Follow operation chain
        current_op = source_graph.value(pc_uri, tb.nextStep)
        self._add_operation_chain(current_op, source_graph, target_graph)
    
    def _add_operation_chain(self, op_uri, source_graph, target_graph):
        """Recursively add operation chain"""
        if op_uri is None:
            return
        
        # Add operation triples
        for p, o in source_graph.predicate_objects(op_uri):
            target_graph.add((op_uri, p, o))
            
            # IMPORTANT: Extract hasInput node triples (role + value)
            if isinstance(o, (URIRef, BNode)) and p == tb.hasInput:
                for ip, io in source_graph.predicate_objects(o):
                    target_graph.add((o, ip, io))
        
        # Follow to next operation
        next_op = source_graph.value(op_uri, tb.nextStep)
        if next_op:
            self._add_operation_chain(next_op, source_graph, target_graph)
    
    def save_policy_checker(self, graph: Graph, output_path: str):
        """Save PolicyChecker to file"""
        graph.serialize(output_path, format='turtle')
        self.logger.info(f"Saved PolicyChecker to {output_path}")

# Initialize planner
planner = SemanticN3Planner(rules_dir="rules_n3_semantic")
print("✓ Semantic N3 Planner initialized")

In [ ]:
# Load SDM
sdm_path = "../../../FederatedComputationalGovernance/SemanticDataModel/sdm.ttl"
sdm_graph = planner.load_sdm(sdm_path)

print(f"✓ Loaded SDM with {len(sdm_graph)} triples")

# Generate PolicyChecker
dataset_id = "UPENN-GBM_clinical_info_v21_timestampcsv"
policy_checker = planner.generate_policy_checkers(sdm_graph, dataset_id)

print(f"✓ Generated PolicyChecker with {len(policy_checker)} triples")

# Save PolicyChecker
output_file = f"policy_checker_{dataset_id}_semantic.ttl"
planner.save_policy_checker(policy_checker, output_file)

print(f"✓ Saved to: {output_file}")

In [ ]:
# Find PolicyCheckers
policy_checkers = list(policy_checker.subjects(RDF.type, tb.PolicyChecker))
print(f"Found {len(policy_checkers)} PolicyCheckers:\n")

for pc in policy_checkers:
    policy_uri = policy_checker.value(pc, tb.accordingTo)
    dataset_uri = policy_checker.value(pc, tb.validates)
    
    print(f"PolicyChecker: {pc}")
    print(f"  According to policy: {policy_uri}")
    print(f"  Validates: {dataset_uri}")
    
    # Trace operations
    current_op = policy_checker.value(pc, tb.nextStep)
    op_count = 0
    
    print(f"\n  Operations:")
    while current_op:
        op_count += 1
        abstract = policy_checker.value(current_op, tb.hasAbstract)
        is_terminal = policy_checker.value(current_op, tb.isTerminal)
        
        print(f"\n  {op_count}. {abstract}")
        print(f"     URI: {current_op}")
        
        # Show inputs with roles
        for input_node in policy_checker.objects(current_op, tb.hasInput):
            role = policy_checker.value(input_node, tb.role)
            value = policy_checker.value(input_node, tb.value)
            print(f"     hasInput: [role=\"{role}\", value=\"{value}\"]")
        
        # Show output
        output = policy_checker.value(current_op, tb.hasOutput)
        if output:
            print(f"     hasOutput: {output}")
        
        if is_terminal:
            print(f"     [TERMINAL]")
            break
        
        current_op = policy_checker.value(current_op, tb.nextStep)
    
    print("\n" + "="*70 + "\n")

In [ ]:
def test_individual_rule(rule_name: str, sdm_graph: Graph):
    """Test a single N3 rule file"""
    rule_path = Path("rules_n3_semantic") / f"{rule_name}.n3"
    
    if not rule_path.exists():
        print(f"❌ Rule not found: {rule_path}")
        return
    
    print(f"Testing rule: {rule_name}")
    print(f"Rule file: {rule_path}\n")
    
    # Create temp directory with single rule
    import tempfile
    import shutil
    
    with tempfile.TemporaryDirectory() as tmpdir:
        # Copy single rule
        shutil.copy(rule_path, Path(tmpdir) / rule_path.name)
        
        # Apply rule
        reasoner = EYEReasoner()
        result = reasoner.apply_rules(sdm_graph, Path(tmpdir))
        
        print(f"✓ Generated {len(result)} triples\n")
        
        # Show sample triples
        print("Sample triples:")
        for i, (s, p, o) in enumerate(result):
            if i >= 10:
                print(f"... ({len(result) - 10} more triples)")
                break
            print(f"  {s}")
            print(f"    {p} → {o}")
    
    return result

# Example: Test LoadData rule
# result = test_individual_rule("LoadData", sdm_graph)

In [ ]:
def create_custom_rule(rule_content: str, rule_name: str = "custom_rule"):
    """Create and test a custom N3 rule"""
    import tempfile
    from pathlib import Path
    
    # Write rule to temp file
    with tempfile.TemporaryDirectory() as tmpdir:
        rule_file = Path(tmpdir) / f"{rule_name}.n3"
        rule_file.write_text(rule_content)
        
        print(f"Testing custom rule: {rule_name}")
        print("="*70)
        print(rule_content)
        print("="*70)
        
        # Apply rule
        reasoner = EYEReasoner()
        result = reasoner.apply_rules(sdm_graph, Path(tmpdir))
        
        print(f"\n✓ Generated {len(result)} triples")
        return result

# Example: Create a rule for anonymization operations
example_rule = """
@prefix tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>.
@prefix ab: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#>.
@prefix odrl: <http://www.w3.org/ns/odrl/2/>.

# Rule: Transform privacy policies into anonymization operations
{
  ?permission odrl:action ab:use ;
              odrl:constraint ?constraint .
  ?constraint odrl:leftOperand ab:privacy ;
              odrl:operator odrl:eq ;
              odrl:rightOperand ?privacyLevel .
  ?permission tb:targetAttribute ?attr .
}
=>
{
  _:anonOp a tb:Operation ;
           tb:hasAbstract ab:Anonymize ;
           tb:hasInput [
               tb:value ab:data ;
               tb:role "previousResult"
           ] ;
           tb:hasInput [
               tb:value ?attr ;
               tb:role "targetAttribute"
           ] ;
           tb:hasInput [
               tb:value ?privacyLevel ;
               tb:role "privacyLevel"
           ] ;
           tb:hasOutput ab:data .
}.
"""

# Uncomment to test:
# result = create_custom_rule(example_rule, "Anonymization")

print("✓ Custom rule template ready")

## 7. Prototype New N3 Rules

Template for creating and testing new transformation rules.

## 6. Test Individual N3 Rules

Test transformation rules independently for prototyping.

## 5. Inspect Generated PolicyChecker Structure

Visualize the role-annotated hasInput structure.

## 4. Load SDM and Generate PolicyChecker

Load the Semantic Data Model and generate PolicyCheckers for the UPENN dataset.

## 3. Semantic N3 Planner Class

Generates PolicyCheckers with role-annotated hasInput from ODRL policies.

## 2. EYE Reasoner Class

The EYE reasoner applies N3 transformation rules to generate PolicyCheckers.

## 1. Imports and Setup